# S4.5 - Duplicates and ambiguous columns after a join
**Date completed:** November 2026  
**Status:** Completed  
**Key concept:** A join can leave duplicate column names and duplicate rows; handle both before the next step

In [0]:
# ============================================================
# S4.5 - Cell 2
# Join Cardinality, Fan-Out, and Ambiguous Columns
#
# Goal:
# 1. See an ambiguous-column error after an expression join.
# 2. Fix ambiguity using:
#    - list-form join key
#    - aliases + explicit select
#    - rename before joining
# 3. Understand join cardinality:
#    - 1:1
#    - 1:N
#    - N:1
#    - N:N
# 4. See exactly how duplicate join keys cause fan-out.
# 5. Detect repeated keys before joining.
# 6. Deduplicate only when uniqueness is a real business rule.
# 7. Practice safe self-join aliasing.
#
# Scope:
# - ambiguous columns
# - duplicate join keys
# - row multiplication / fan-out
# - cardinality
# - self-join aliases
#
# Deferred:
# - deterministic latest-row selection with Window
#   -> S4.6 / S4.7
# ============================================================

import pyspark.sql.functions as F


print("=== S4.5: JOIN CARDINALITY AND AMBIGUOUS COLUMNS ===")
print()

# ============================================================
# Step 1 - Controlled Source Data
# ============================================================

orders = spark.createDataFrame(
    [
        (1, 10, 500),
        (2, 20, 300),
        (3, 10, 200)
    ],
    """
    order_id INT,
    id INT,
    amount INT
    """
)

customers = spark.createDataFrame(
    [
        (10, "Asha"),
        (20, "Ben")
    ],
    """
    id INT,
    name STRING
    """
)

print("=== STEP 1: SOURCE ===")

orders.orderBy("order_id").show()
customers.orderBy("id").show()

In [0]:
# ============================================================
# Step 2 - Ambiguous Column Problem
#
# Expression join:
#
# orders.id == customers.id
#
# Both id columns remain in the result.
#
# Prediction:
#
# j.columns
# → contains two columns named "id"
#
# select("id")
# → ambiguous
# ============================================================

cond = (orders["id"] == customers["id"])

j = (orders.join(customers, cond, "inner"))

print("=== STEP 2: AMBIGUOUS COLUMN ===")

print("RESULT columns:", j.columns)

print("RESULT number of columns named id:", j.columns.count("id"))

try:

    j.select("id").collect()

    print("RESULT select('id'): worked (unexpected)")

except Exception as e:

    print(
        "RESULT select('id'):",
        type(e).__name__,
        "|",
        str(e).splitlines()[0][:160]
    )


In [0]:
# ============================================================
# Step 3 - Fix A: List-Form Join Key
#
# Because both DataFrames use the same key name:
#
# on="id"
#
# Spark returns the common join key once.
# ============================================================

fix_a = (orders.join(customers,on="id",how="inner"))

print()
print("=== STEP 3: FIX A - LIST FORM ===")

print("RESULT columns:", fix_a.columns)

print("RESULT number of id columns:", fix_a.columns.count("id"))


In [0]:
# ============================================================
# Step 4 - Fix B: Aliases + Qualified Columns
#
# Expression join can retain both ids.
#
# Aliases let us explicitly identify which side each
# column came from.
# ============================================================

o = orders.alias("o")
c = customers.alias("c")

fix_b = (o.join(c,F.col("o.id") == F.col("c.id"), "inner")

    .select(
        F.col("o.order_id"),
        F.col("o.id").alias("order_cust_id"),
        F.col("c.id").alias("customer_id"),
        F.col("o.amount"),
        F.col("c.name")))

print()
print("=== STEP 4: FIX B - ALIASES ===")

print("RESULT columns:", fix_b.columns)


In [0]:
# ============================================================
# Step 5 - Fix C: Rename Before Joining
#
# Rename the right-side id before constructing the join.
#
# This prevents the name collision.
# ============================================================

customers_renamed = (customers.withColumnRenamed("id","customer_id"))

fix_c = (orders.join(customers_renamed,orders["id"] == customers_renamed["customer_id"],"inner"))

print()
print("=== STEP 5: FIX C - RENAME ===")

print("RESULT columns:",fix_c.columns)


In [0]:
# ============================================================
# Step 6 - Duplicate Column Names and Table Write
#
# Prediction:
#
# j contains two columns named "id".
#
# Spark can hold this DataFrame temporarily, but writing it
# as a table requires unique column names.
#
# Therefore saveAsTable() is EXPECTED to fail with an error
# such as COLUMN_ALREADY_EXISTS.
#
# The exception is intentionally caught so the notebook
# can continue.
# ============================================================

print()
print("=== STEP 6: EXPECTED WRITE FAILURE - DUPLICATE COLUMNS ===")

dup_name = "workspace.default.s4_5_dup_test"

try:

    (
        j.write
        .mode("overwrite")
        .saveAsTable(dup_name)
    )

    print("RESULT: write succeeded unexpectedly")

except Exception as e:

    print("RESULT: expected write failure")

    print("Error type:",type(e).__name__)

    print("Error:",str(e).splitlines()[0][:160])

# Cleanup in case a table was partially or unexpectedly created.
spark.sql(f"DROP TABLE IF EXISTS {dup_name}")

In [0]:
# ============================================================
# Step 7 - Cardinality and Fan-Out
#
# IMPORTANT:
#
# Fan-out comes from join-key cardinality.
#
# For a given key:
#
# joined pairs
# =
# left key count × right key count
#
#
# Current LEFT:
#
# id 10 → 2 orders
# id 20 → 1 order
#
#
# New RIGHT:
#
# id 10 → 2 customer rows
# id 20 → 1 customer row
#
#
# Therefore:
#
# id 10:
# 2 × 2 = 4 joined rows
#
# id 20:
# 1 × 1 = 1 joined row
#
# TOTAL:
#
# 4 + 1 = 5 rows
#
# This is N:N behaviour for id 10.
# ============================================================

cust_dup = spark.createDataFrame(
    [
        (10, "Asha"),
        (10, "Asha B"),
        (20, "Ben")
    ],
    """
    id INT,
    name STRING
    """
)


fan = (orders.join(cust_dup,on="id",how="inner"))

orders_count = (orders.count())

fan_count = (fan.count())

print()
print("=== STEP 7: FAN-OUT ===")

print(f"RESULT source orders : "f"{orders_count}")

print(f"RESULT predicted join: "f"5")

print(f"RESULT actual join   : "f"{fan_count}")

print(f"RESULT prediction matched: "f"{fan_count == 5}")

fan.orderBy(
    "id",
    "order_id",
    "name"
).show()

In [0]:
# ============================================================
# Step 8 - Explain Cardinality Per Key
#
# Count how many records each key has on each side.
#
# For an INNER JOIN:
#
# expected pairs for each matching key
# =
# left_count × right_count
#
# This makes fan-out measurable before the join.
# ============================================================

left_key_counts = (
    orders
    .groupBy("id")
    .count()
    .withColumnRenamed(
        "count",
        "left_count"
    )
)


right_key_counts = (
    cust_dup
    .groupBy("id")
    .count()
    .withColumnRenamed(
        "count",
        "right_count"
    )
)


cardinality = (
    left_key_counts.join(right_key_counts,on="id",how="inner")
    .withColumn(
        "expected_join_rows",
        F.col("left_count") * F.col("right_count")))

print()
print("=== STEP 8: CARDINALITY BY KEY ===")

cardinality.orderBy("id").show()

expected_join_rows = (
    cardinality
    .agg(F.sum("expected_join_rows").alias("expected")).first()["expected"])

print(
    f"RESULT expected from cardinality: "
    f"{expected_join_rows}")

print(
    f"RESULT actual joined rows       : "
    f"{fan_count}")

print(
    f"RESULT reconciliation           : "
    f"{expected_join_rows == fan_count}")


In [0]:
# ============================================================
# Step 9 - Detect Repeated Keys BEFORE Joining
#
# Check BOTH sides.
#
# Duplicate keys on either side matter.
#
# Possible relationships:
#
# 1:1
# → one left row, one right row
#
# N:1
# → multiple left rows, one right row
#
# 1:N
# → one left row, multiple right rows
#
# N:N
# → multiple rows on both sides
#
# N:N creates multiplicative fan-out.
# ============================================================

left_duplicates = (
    orders
    .groupBy("id")
    .count()
    .filter(
        F.col("count") > 1
    )
)


right_duplicates = (
    cust_dup
    .groupBy("id")
    .count()
    .filter(
        F.col("count") > 1
    )
)


left_duplicate_result = [
    (r["id"], r["count"])
    for r in left_duplicates.orderBy("id").collect()]

right_duplicate_result = [
    (r["id"], r["count"])
    for r in right_duplicates.orderBy("id").collect()]

print()
print("=== STEP 9: REPEATED KEYS ===")

print(
    "RESULT repeated left keys :",
    left_duplicate_result)

print(
    "RESULT repeated right keys:",
    right_duplicate_result
)

In [0]:
# ============================================================
# Step 10 - Deduplicate Right Side
#
# IMPORTANT:
#
# Deduplication is ONLY correct when the business rule says:
#
# "there must be one right-side row per id"
#
# Do NOT blindly deduplicate just to make the row count match.
#
# In this tiny controlled example we use MIN(name) only to
# demonstrate one-row-per-id behavior.
#
# MIN(name) is NOT a generic production survivor rule.
#
# In a real system the rule might be:
#
# - latest updated_at
# - highest version
# - active record
# - preferred source
#
# Deterministic latest-row selection will be learned later
# with Window functions.
# ============================================================

dedup_demo = (
    cust_dup
    .groupBy("id")
    .agg(
        F.min("name").alias("name")
    )
)

fixed = (
    orders
    .join(
        dedup_demo,
        on="id",
        how="inner"))

fixed_count = (
    fixed.count())

print()
print("=== STEP 10: DEDUP DEMO ===")

print(
    f"RESULT right rows before : "
    f"{cust_dup.count()}")

print(
    f"RESULT right rows after  : "
    f"{dedup_demo.count()}")

print(
    f"RESULT joined rows       : "
    f"{fixed_count}")

print(
    f"RESULT joined == orders  : "
    f"{fixed_count == orders_count}")

In [0]:
# ============================================================
# Step 11 - Self Join and Aliases
#
# A self join uses the same DataFrame on both sides.
#
# Without aliases, column references become difficult or
# ambiguous because both sides come from the same schema.
#
# Example:
#
# employee.manager_id
#       →
# employee.emp_id
# ============================================================

employees = spark.createDataFrame(
    [
        (1, "CEO", None),
        (2, "Asha", 1),
        (3, "Ben", 1),
        (4, "Chen", 2)
    ],
    """
    emp_id INT,
    name STRING,
    manager_id INT
    """
)


e = employees.alias("e")
m = employees.alias("m")


employee_manager = (
    e
    .join(
        m,
        F.col("e.manager_id")
        ==
        F.col("m.emp_id"),
        "left"
    )
    .select(
        F.col("e.emp_id"),
        F.col("e.name").alias(
            "employee_name"
        ),
        F.col("e.manager_id"),
        F.col("m.name").alias(
            "manager_name"
        )
    )
)

print()
print("=== STEP 11: SELF JOIN ===")

employee_manager.orderBy("emp_id").show()

In [0]:
# ============================================================
# Step 12 - Controlled Checkpoint
# ============================================================

print()
print("=== STEP 12: S4.5 CHECKPOINT ===")


print(
    "Expression join id columns:",
    j.columns.count("id")
)


print(
    "List-form join id columns:",
    fix_a.columns.count("id")
)


print(
    "Fan-out predicted:",
    5
)


print(
    "Fan-out actual   :",
    fan_count
)


print(
    "After dedup rows :",
    fixed_count
)


print(
    "Source order rows:",
    orders_count
)


print(
    "Cardinality reconciliation:",
    expected_join_rows == fan_count
)


# Controlled assertions.

assert (
    j.columns.count("id")
    ==
    2
)


assert (
    fix_a.columns.count("id")
    ==
    1
)

assert (
    fan_count
    ==
    5
)


assert (
    expected_join_rows
    ==
    fan_count
)


assert (
    fixed_count
    ==
    orders_count
)

print("RESULT all controlled S4.5 checks passed.")

In [0]:
# ============================================================
# Step 13 - Final Learning Summary
# ============================================================

print()
print("=== FINAL LEARNING SUMMARY ===")


print("""
AMBIGUOUS COLUMNS

Expression join:
left.id == right.id

→ both id columns can remain
→ select("id") can become ambiguous


FIXES

1. Same-name equi-key:
   on="id"

2. Alias both sides:
   left.alias("l")
   right.alias("r")

3. Rename before joining


CARDINALITY

1:1
→ one match on each side

N:1
→ many left rows can match one right row

1:N
→ one left row can match many right rows

N:N
→ multiple rows on both sides
→ multiplicative fan-out


FAN-OUT RULE

For each matching key:

joined rows
=
left key count
×
right key count


IMPORTANT

More rows after a join do not automatically mean
Spark duplicated data incorrectly.

The join may be correctly reflecting the cardinality
of the source keys.


DEDUPLICATION

Do not deduplicate only to make row counts look correct.

First determine whether uniqueness is actually required
by the business relationship.


SELF JOIN

Always use aliases so each side can be referenced
unambiguously.


CORE ARCHITECT RULE

Before joining, know the expected relationship:

1:1
N:1
1:N
N:N

Then reconcile the actual result against that expected
grain and cardinality.
""")


print(
    "=== S4.5 CELL 2 COMPLETED ==="
)

# S4.5 — Key Takeaways: Duplicates and Ambiguous Columns After a Join

## 1. The Problem

An expression join such as:

```python id="9ad1lf"
left.id == right.id
```

can retain both `id` columns.

The DataFrame can exist with duplicate column names, but downstream operations may become ambiguous or invalid.

Observed in our workspace:

```text id="jsjdli"
select("id")
→ AnalysisException [AMBIGUOUS_REFERENCE]

write as a table
→ AnalysisException [COLUMN_ALREADY_EXISTS]
```

---

## 2. Three Ways to Handle Duplicate Column Names

| Fix | Code | Result |
|---|---|---|
| Same-name join key | `on="id"` or `on=["a", "b"]` | common join key appears once |
| Alias both sides | `o = orders.alias("o")`, `c = customers.alias("c")` | both sides remain explicitly addressable |
| Rename before join | `customers.withColumnRenamed("id", "customer_id")` | avoids the naming clash |

For list-form joins, the named keys must exist with the same names on both DataFrames.

For self joins, aliases should be used when both sides need to be referenced clearly:

```python id="4wc8og"
e = employees.alias("e")
m = employees.alias("m")
```

---

## 3. Cardinality and Fan-Out

For each matching key:

```text id="a66ad7"
joined rows
=
left rows for that key
×
right rows for that key
```

| Relationship | Meaning |
|---|---|
| `1:1` | one row on each side |
| `N:1` | many left rows match one right row |
| `1:N` | one left row matches many right rows |
| `N:N` | multiple rows exist on both sides and can multiply |

More rows after a join are not automatically a Spark error.

The join may be correctly reflecting the source-key cardinality.

---

## 4. Check Cardinality Before Joining

Before an important join:

- count repeated keys on the left
- count repeated keys on the right
- understand the expected relationship
- predict possible row multiplication
- reconcile the actual joined row count

For an inner join, per-key counts can be used to calculate:

```text id="c4nyl7"
expected rows
=
sum(left_count × right_count)
```

over matching keys.

---

## 5. Deduplicating

Do not deduplicate only to make a row count match.

First decide whether the business relationship actually requires:

```text id="iv0mzg"
one right-side row per business key
```

If one row must survive, the business rule must specify which row wins.

Examples:

```text id="ln12gb"
latest updated_at
highest version
active record
preferred source
```

Our controlled example used:

```python id="j59ajd"
F.min("name")
```

only to demonstrate one-row-per-key behavior.

It is **not** a general production survivor rule.

Deterministic survivor selection will be covered with:

```text id="vhtwbj"
S4.6 → Window specification
S4.7 → row_number / ranking
```

---

## 6. Observed in Our Workspace

Environment:

```text id="f7d2ai"
Spark   : 4.2.0
Compute : Databricks Serverless
```

### Ambiguous Columns

```text id="qsgabp"
Expression join columns:
['order_id', 'id', 'amount', 'id', 'name']

Columns named id:
2
```

Attempt:

```python id="m2vdvb"
select("id")
```

Observed:

```text id="69ymzn"
AnalysisException
[AMBIGUOUS_REFERENCE]
```

---

### Fix A — Same-Name Key

```python id="zcbkl5"
on="id"
```

Observed:

```text id="74xydf"
['id', 'order_id', 'amount', 'name']
```

Only one `id` column remained.

---

### Fix B — Aliases

Observed:

```text id="qycnnr"
['order_id',
 'order_cust_id',
 'customer_id',
 'amount',
 'name']
```

Both original keys remained available under clear names.

---

### Fix C — Rename Before Join

Observed:

```text id="7djok8"
['order_id',
 'id',
 'amount',
 'customer_id',
 'name']
```

No naming clash remained.

---

### Write With Duplicate Column Names

Attempting to write the unresolved expression-join result produced:

```text id="81hlju"
AnalysisException
[COLUMN_ALREADY_EXISTS]
```

This confirmed that the duplicate `id` names had to be resolved before this table write in our workspace.

---

## 7. Fan-Out Checkpoint

Source orders:

```text id="szp9j8"
3 rows
```

For key `10`:

```text id="rtvwgz"
left count  = 2
right count = 2

2 × 2 = 4 rows
```

For key `20`:

```text id="2mshl9"
left count  = 1
right count = 1

1 × 1 = 1 row
```

Therefore:

```text id="c7vcq6"
predicted joined rows
=
4 + 1
=
5
```

Observed:

```text id="c7foyp"
predicted = 5
actual    = 5
```

The cardinality reconciliation matched.

---

## 8. Repeated Keys

Observed:

```text id="hs9e42"
Left:
id 10 → 2 rows

Right:
id 10 → 2 rows
```

Therefore key `10` had an `N:N` relationship in this controlled example.

That explains its multiplicative fan-out.

---

## 9. Dedup Demonstration

Right-side rows before:

```text id="45o44p"
3
```

After reducing to one row per `id`:

```text id="g6gu2q"
2
```

Joined rows after that reduction:

```text id="rvgn6q"
3
```

Source orders:

```text id="5i271o"
3
```

Therefore:

```text id="2erqk4"
joined rows == source orders
→ True
```

This result is valid only because our controlled demonstration intentionally reduced the right side to one row per key.

It does not mean deduplication is always the correct response to fan-out.

---

## 10. Not Yet Verified

The checkpoint evidence shared so far does not include the output of the self-join example.

Therefore the self-join alias concept is understood but not yet recorded as an observed workspace result.

The failed-write cleanup command was executed, but we did not separately query the catalog afterward to prove whether any residual test table existed before cleanup.

---

## 11. Interview Q&A

### Q1. Why does `select("id")` fail after some expression joins?

Because both sides retained a column named `id`, so Spark cannot determine which one `"id"` refers to.

---

### Q2. How can duplicate join-key names be handled?

Use:

```text id="txjylr"
same-name list/string join
aliases and qualified columns
rename before joining
```

depending on the relationship and desired output schema.

---

### Q3. A join returns more rows than the left table. Is that automatically a bug?

No.

Check the cardinality of the join keys.

For each key:

```text id="j6d20u"
left count × right count
```

determines how many joined pairs can be produced.

---

### Q4. How can fan-out be predicted before a join?

Group each side by the join key, count rows per key, and inspect repeated keys.

For an inner join, reconcile the expected joined pairs using the per-key counts.

---

### Q5. Should duplicate keys always be removed before joining?

No.

Duplicate keys may represent valid `1:N`, `N:1`, or `N:N` business relationships.

Deduplicate only when the business rule requires uniqueness.

---

## 12. Core Understanding

```text id="6exkt1"
Ambiguous columns
→ schema/reference problem

Duplicate keys
→ cardinality problem

Fan-out
→ possible consequence of cardinality
```

Before joining, know:

```text id="1b4y89"
expected grain
join key
expected cardinality
expected row-count behavior
```

Then reconcile the actual result against those expectations.

**S4.5 — checkpoint pending self-join validation.**